In [ ]:
#install folium
!pip install folium
#add libraries for tabular and vector data and for plotting
import folium
import json
from google.colab import files
#mount the drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
#define file paths
path_1980 = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/pre2k.geojson'
path_2002 = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/post2k.geojson'
path_pct = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/pctchange_gdf.geojson'

In [ ]:
# Initialize Folium map centered on Colorado
m = folium.Map(location=[39.0, -105.5], zoom_start=7, tiles="OpenStreetMap")

In [ ]:
# Styling function for NIA_mean data
def style_function(feature):
   fill_color = 'grey' # Default color

   # default to 0 if None
   d = feature['properties'].get('NIA_mean', 0)

   if d is not None: # Ensure d is not None before comparing
       if d > 18: # Example thresholds for NIA_mean - adjusted based on original NIA styling
           fill_color = 'rgb(0, 90, 50)'
       elif d > 15:
           fill_color = 'rgb(39, 116, 80)'
       elif d > 10:
           fill_color = 'rgb(79, 142, 111)'
       elif d > 6:
           fill_color = 'rgb(118, 169, 141)'
       elif d > 4:
           fill_color = 'rgb(158, 195, 172)'
       elif d > 1.5:
           fill_color = 'rgb(197, 221, 202)'
       else:
           fill_color = 'rgb(237, 248, 233)'

   return {
   'fillColor': fill_color,
   'color': 'white',
   'weight': 1,
   'fillOpacity': 1
         }

# Load and add 1980–2002 data
with open(path_1980) as f:
 data_1980 = json.load(f)
layer_1980 = folium.FeatureGroup(name='Irrigated Acres 1982–1997')
folium.GeoJson(
 data_1980,
 style_function=style_function,
 tooltip=folium.GeoJsonTooltip(
 fields=['COUNTY', 'NIA_mean'],
 aliases=['COUNTY', 'Percent of Land Irrigated']
 )
).add_to(layer_1980)
layer_1980.add_to(m)

# Load and add 2002–2024 data
with open(path_2002) as f:
 data_2002 = json.load(f)
layer_2002 = folium.FeatureGroup(name='Irrigated Acres 2002–2022')
folium.GeoJson(
 data_2002,
 style_function=style_function,
 tooltip=folium.GeoJsonTooltip(
 fields=['COUNTY', 'NIA_mean'],
 aliases=['COUNTY', 'Percent of Land Irrigated']
 )
).add_to(layer_2002)
layer_2002.add_to(m)

In [ ]:
# Updated legend for Irrigated Acres
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; width: 200px; height: 180px;
  border:2px solid grey; z-index:9999; font-size:14px;
  background-color: rgba(255, 255, 255, 0.9); padding: 10px;">
<b>Percent of Land Irrigated</b><br>
<i style="background: rgb(237, 248, 233);width:15px;height:15px;float:left;margin-right:8px;"></i> 0–1.5%<br>
<i style="background: rgb(197, 221, 202);width:15px;height:15px;float:left;margin-right:8px;"></i> 1.5–4%<br>
<i style="background: rgb(158, 195, 172);width:15px;height:15px;float:left;margin-right:8px;"></i> 4–6%<br>
<i style="background: rgb(118, 169, 141);width:15px;height:15px;float:left;margin-right:8px;"></i> 6–10%<br>
<i style="background: rgb(79, 142, 111);width:15px;height:15px;float:left;margin-right:8px;"></i> 10–15%<br>
<i style="background: rgb(39, 116, 80);width:15px;height:15px;float:left;margin-right:8px;"></i> 15–18%<br>
<i style="background: rgb(0, 90, 50);width:15px;height:15px;float:left;margin-right:8px;"></i> 18%+<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Add control panel
folium.LayerControl(collapsed=False).add_to(m)

In [ ]:
# Save and download
m.save('colorado_nia_map.html')
files.download('colorado_nia_map.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# display the map
m

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# create a new folium map displaying the percent change of irrigated acres
#new map
mpct = folium.Map(location=[39.0, -105.5], zoom_start=7, tiles="OpenStreetMap")
# Styling function for percentage change data
def style_function_pct(feature):
  fill_color = 'grey' # Default color
  d = feature['properties'].get('ia_pctchange', None) # Get percent change, default to None if key is missing

  # Handle None values explicitly or assign default color
  if d is None:
      return {
          'fillColor': 'grey', # Color for incomplete data
          'color': 'white',
          'weight': 1,
          'fillOpacity': 0.7 # Adjust opacity for the change layer
      }

  # Now d is guaranteed to be a number (or 0 if it was None, although handling None explicitly above is better)
  # Using the ranges and colors from the legend_pct_html
  if d > 0: # Ranges for increases (not in legend, but assuming white/lightest color for 0-15% decrease)
      fill_color = 'rgb(237, 248, 233)' # Using the lightest color from NIA_mean legend for increases/small decreases not explicitly in pct change legend
  elif d > -15: # 0-15% Decrease (Light Brown)
      fill_color = 'rgb(255, 245, 230)'
  elif d > -30: # 15-30% Decrease
      fill_color = 'rgb(235, 195, 154)'
  elif d > -40: # 30-40% Decrease
      fill_color = 'rgb(214, 146, 83)'
  elif d > -60: # 40-60% Decrease
      fill_color = 'rgb(175, 107, 47)'
  elif d > -80: # 60-80% Decrease
      fill_color = 'rgb(116, 77, 48)'
  elif d > -95: # 80-95% Decrease (corresponds to -95 to -80 in style, but legend says 80-95%)
      fill_color = 'rgb(89, 68, 57)' # Dark brown
  else: # < -95% (Corresponds to 95%+ decrease)
       fill_color = 'darkred' # Using dark red to highlight extreme decrease, not in legend color scale but for values < -95


  return {
      'fillColor': fill_color,
      'color': 'white',
      'weight': 1,
      'fillOpacity': 1
  }

# Load and add percentage change data
with open(path_pct) as f:
    data_pct = json.load(f)

layer_pct = folium.FeatureGroup(name='Percent Change in Irrigated Acres')
folium.GeoJson(
    data_pct,
    style_function=style_function_pct,
    tooltip=folium.GeoJsonTooltip(
        fields=['ia_pctchange'],
        aliases=['Percent Change']
    )
).add_to(layer_pct)
layer_pct.add_to(mpct)

# Add legend for percent change
legend_pct_html = """
<div style="position: fixed; bottom: 40px; left: 40px; width: 200px; height: 180px;
  border:2px solid grey; z-index:9999; font-size:14px;
  background-color: rgba(255, 255, 255, 0.9); padding: 10px;">
<b>Percent Change in Irrigated Acres</b><br>
<i style="background: grey;width:15px;height:15px;float:left;margin-right:8px;"></i> Incomplete Data<br>
<i style="background: rgb(255, 245, 230);width:15px;height:15px;float:left;margin-right:8px;"></i> 0-15% Decrease<br>
<i style="background: rgb(235, 195, 154);width:15px;height:15px;float:left;margin-right:8px;"></i> 15-30% Decrease<br>
<i style="background: rgb(214, 146, 83);width:15px;height:15px;float:left;margin-right:8px;"></i> 30-40% Decrease<br>
<i style="background: rgb(175, 107, 47);width:15px;height:15px;float:left;margin-right:8px;"></i> 40-60% Decrease<br>
<i style="background: rgb(116, 77, 48);width:15px;height:15px;float:left;margin-right:8px;"></i> 60-80% Decrease<br>
<i style="background: rgb(89, 68, 57);width:15px;height:15px;float:left;margin-right:8px;"></i> 80-95% Decrease<br>
</div>
"""
mpct.get_root().html.add_child(folium.Element(legend_pct_html))

# Update the control panel to include the new layer
folium.LayerControl(collapsed=False).add_to(mpct)

# Display the map
mpct

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Save and download the percent change map
mpct.save('colorado_pctchange_irrigatedacres.html')
files.download('colorado_pctchange_irrigatedacres.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>